In [ ]:
import csv
import os
import uuid
import tqdm
import geopandas as gpd
import numpy as np
from PIL import Image
import rasterio
from rasterio.crs import CRS
from rasterio.warp import transform_bounds
from shapely.geometry import box


def tile_raster_by_landmass(
    raster_path: str,
    shapefile_path: str,
    output_dir: str,
    csv_out_path: str,
    tile_size: int = 224,
):
    """Tiles a GeoTIFF into tile_size x tile_size PNGs if they overlap with a landmass shapefile.

    Saves a CSV containing unique IDs, filenames, and centroid Lat/Lon (EPSG:4326).
    """
    os.makedirs(output_dir, exist_ok=True)

    # 1. Load Landmass Shapefile
    print("Loading landmass shapefile...")
    land_gdf = gpd.read_file(shapefile_path)

    # Ensure shapefile is valid and has spatial index
    land_gdf = land_gdf[land_gdf.geometry.notnull()]
    land_sindex = land_gdf.sindex

    # Open GeoTIFF
    with rasterio.open(raster_path) as src:
        width = src.width
        height = src.height
        raster_crs = src.crs
        transform = src.transform

        print(
            f"Raster Size: {width}x{height} | Bands: {src.count} | CRS: {raster_crs}"
        )

        # Ensure Land Geometry CRS matches Raster CRS for fast bounding-box checks
        if land_gdf.crs != raster_crs:
            print(
                f"Reprojecting landmass geometry from {land_gdf.crs} to {raster_crs}..."
            )
            land_gdf_reprojected = land_gdf.to_crs(raster_crs)
        else:
            land_gdf_reprojected = land_gdf

        # CSV Preparation
        records = []

        # Target CRS for CSV output (Lat/Lon = EPSG:4326)
        wgs84_crs = CRS.from_epsg(4326)

        # 2. Iterate through raster grid in tile_size steps
        total_tiles_processed = 0
        land_tiles_saved = 0

        print("Starting tiling loop...")

        for y in tqdm.tqdm(range(0, height - tile_size + 1, tile_size)):
            for x in range(0, width - tile_size + 1, tile_size):
                total_tiles_processed += 1

                # Pixel window coordinates to Geographic/Projected Bounding Box
                # Bounds: (minx, miny, maxx, maxy)
                tile_win_bounds = rasterio.windows.bounds(
                    rasterio.windows.Window(x, y, tile_size, tile_size),
                    transform,
                )
                tile_box = box(*tile_win_bounds)

                # 3. Spatial Intersect Check against Landmass Shapefile using Spatial Index
                possible_matches_index = list(
                    land_sindex.intersection(tile_box.bounds)
                )
                possible_matches = land_gdf_reprojected.iloc[
                    possible_matches_index
                ]

                # Check actual intersection
                if not possible_matches.intersects(tile_box).any():
                    continue  # Skip tile - pure ocean / outside landmass

                # 4. Read RGB Image Data for Tile Window
                window = rasterio.windows.Window(x, y, tile_size, tile_size)
                # Read 3 bands (RGB)
                rgb_data = src.read([1, 2, 3], window=window)

                # Transpose from (Bands, Height, Width) to (Height, Width, Bands) for PIL Image
                rgb_array = np.transpose(rgb_data, (1, 2, 0)).astype(np.uint8)

                # Skip completely black/nodata filled tiles if necessary
                if not np.any(rgb_array):
                    continue

                # 5. Calculate Centroid (Lat / Lon in EPSG:4326)
                centroid = tile_box.centroid
                if raster_crs != wgs84_crs:
                    # Reproject centroid to WGS84 (Lat/Lon)
                    cx_bounds = transform_bounds(
                        raster_crs,
                        wgs84_crs,
                        centroid.x,
                        centroid.y,
                        centroid.x,
                        centroid.y,
                    )
                    lon, lat = cx_bounds[0], cx_bounds[1]
                else:
                    lon, lat = centroid.x, centroid.y

                # 6. Generate Unique ID & File Output
                tile_id = str(uuid.uuid4())
                filename = f"{tile_id}.png"
                out_png_path = os.path.join(output_dir, filename)

                # Save Image using PIL
                img = Image.fromarray(rgb_array)
                img.save(out_png_path, format="PNG")

                # Store metadata record
                records.append(
                    {
                        "tile_id": tile_id,
                        "filename": filename,
                        "lat": round(lat, 6),
                        "lon": round(lon, 6),
                        "col_off": x,
                        "row_off": y,
                    }
                )

                land_tiles_saved += 1

                if land_tiles_saved % 1000 == 0:
                    print(f"Saved {land_tiles_saved} land tiles...")

        # 7. Write Index CSV
        print(f"Saving metadata CSV to {csv_out_path}...")
        with open(csv_out_path, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=[
                    "tile_id",
                    "filename",
                    "lat",
                    "lon",
                    "col_off",
                    "row_off",
                ],
            )
            writer.writeheader()
            writer.writerows(records)

        print(f"\nProcessing Complete!")
        print(f"Total Grid Windows Tested: {total_tiles_processed}")
        print(f"Total Land Tiles Saved: {land_tiles_saved}")


if __name__ == "__main__":
    # Define File Paths
    INPUT_GEOTIFF = rf"E:\Data\satclip\world_rgb\blue_marble_global_stitched.tif"
    LAND_SHAPEFILE = rf"E:\Data\Global\World\land-poly\land_polygons.shp"
    OUTPUT_FOLDER = RF"E:\Data\satclip\world_rgb\tiles"
    OUTPUT_CSV = RF"E:\Data\satclip\world_rgb\tiles_META.CSV"

    tile_raster_by_landmass(
        raster_path=INPUT_GEOTIFF,
        shapefile_path=LAND_SHAPEFILE,
        output_dir=OUTPUT_FOLDER,
        csv_out_path=OUTPUT_CSV,
        tile_size=224,
    )

Loading landmass shapefile...
Raster Size: 86400x43200 | Bands: 3 | CRS: EPSG:4326
Starting tiling loop...


  7%|▋         | 14/192 [01:47<1:18:31, 26.47s/it]

Saved 1000 land tiles...


  9%|▉         | 17/192 [07:41<4:19:23, 88.93s/it]